In [563]:
!git clone https://github.com/gkianfar/TIHM-Dataset-Visualization.git
!git clone https://github.com/gkianfar/1234llm.git

fatal: destination path 'TIHM-Dataset-Visualization' already exists and is not an empty directory.
Cloning into '1234llm'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 40 (delta 17), reused 6 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 3.01 MiB | 8.43 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [562]:
!rm -r /content/1234llm

Load TIHM Dataset

In [2]:
%cd /content/TIHM-Dataset-Visualization
from utils import load_datasets
# Load dataset
activity_df, physiology_df, sleep_df, labels_df, demographics_df =\
 load_datasets('/content/TIHM-Dataset-Visualization/Data')

/content/TIHM-Dataset-Visualization


Load Modal

*Due to the large memory requirement to load the original Llama model, we load the light version.*




In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from huggingface_hub import login

device = 'cpu'

# Load TinyLlama model and tokenizer
model_name = "meta-llama/Llama-3.2-1B"
context_length = 128000

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map=device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [26]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Fix 1: Set pad token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def prepare_prompt(question, instruction=None, context=None, external_information=None,
                   INSTRUCTION_LABEL='Instruction: ',
                   KNOWLEDGE_LABEL='Knowledge: ',
                   CONTEXT_LABEL='Context: ',
                   QUESTION_LABEL='Question: ',
                   ANSWER_LABEL='Answer: '):
    """
    Prepares a formatted prompt and generates an answer using the model.

    Parameters:
        question (str): The main question to be answered.
        instruction (str, optional): Specific instructions for answering.
        context (str, optional): Additional context for the prompt.
        external_information (str, optional): Supplementary knowledge.

    Returns:
        str: The generated answer.
    """
    if not question:
        raise ValueError("The 'question' parameter cannot be empty.")

    # Formulate the prompt
    prompt = ""

    if instruction:
        prompt += f"{INSTRUCTION_LABEL}\n{instruction}\n\n"

    if external_information:
        prompt += f"{KNOWLEDGE_LABEL}\n{external_information}\n\n"

    if context:
        prompt += f"{CONTEXT_LABEL}\n{context}\n\n"

    prompt += f"{QUESTION_LABEL}\n{question}\n\n{ANSWER_LABEL}\n"

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

    # Generate an answer
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode the output
    answer = tokenizer.decode(
        output_ids[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return answer

# Fix 5: Corrected function name (typo fix)
question = """You are a virtual tour guide from 1901. You have tourists visiting Eiffel Tower. Describe Eiffel Tower to your audience. Begin with
1. Why it was built
2. Then by how long it took them to build
3. Where were the materials sourced to build
4. Number of people it took to build
5. End it with the number of people visiting the Eiffel tour annually in the 1900's, the amount of time it completes a full tour and why so many people visit this place each year.
Make your tour funny by including 1 or 2 funny jokes at the end of the tour.

"""

answer = prepare_prompt(question)  # Fixed function name
print("Answer:", answer)

Answer: Why it was built:
The tower is made up entirely out of iron which can be recycled.

It was constructed within two years after its completion.


When it began construction? In mid-19th century when France became an industrial nation


Where did they source their raw material for building?
They used scrap metal that had been discarded during manufacturing process



How much money did it take to construct the Eiffel Tour?


*Reloading datasets and examine the performance with few shot in-context learning*

Here, we load the tabular dataset for agitation, comprising context data and features.

In [99]:
import pandas as pd
import numpy as np

# Set dataset path
data_path = '/content/drive/MyDrive/datasets/dataset_clean0_tm1_th0.002_occ15_seed2025_neg1_fill0.csv'

# Load data into a dataframe
df = pd.read_csv(data_path)
#df

In [143]:
# Split to two sets for in-context learning and test

seed_id = 2025  # Ensure it's an integer

# Set seed for reproducibility
np.random.seed(seed_id)
torch.manual_seed(seed_id)

# Split to train and test
ratio = 0.8  # Ratio for splitting to train and test
total_sample_size = df.shape[0]  # Total number of samples
train_size = int(total_sample_size * ratio)  # Train dataset size
test_size = total_sample_size - train_size  # Test dataset size

# Randomly select training sample indices
train_sample_ids = np.random.choice(np.arange(total_sample_size), size=train_size, replace=False)
test_sample_ids = sorted(list(set(np.arange(total_sample_size)) - set(train_sample_ids)))

# Train and test datasets
train_dataset = df.iloc[train_sample_ids].copy()
test_dataset = df.drop(index=train_sample_ids).copy()


In [144]:
# Construct columns for context_columns
context_columns = ['patient_id','start_time','end_time','week_day','gender','normal_samples']

# Set suffixes to filter
specific_strings = ["_change_mean", "_count_mean"]
for ss in specific_strings:
  # Filter out column names that end with the specific string
  context_columns.extend([col for col in df.columns if col.endswith(ss)])
feature_columns = ['change_count', 'Fridge Door', 'Kitchen','Front Door', 'Bedroom', 'Back Door', 'Bathroom', 'Lounge', 'Hallway']


# print the column names
print(f'context:\n{context_columns}')
print(f'features:\n{feature_columns}')

context:
['patient_id', 'start_time', 'end_time', 'week_day', 'gender', 'normal_samples', 'normal_change_mean', 'normal_Fridge Door_count_mean', 'normal_Kitchen_count_mean', 'normal_Front Door_count_mean', 'normal_Bedroom_count_mean', 'normal_Back Door_count_mean', 'normal_Bathroom_count_mean', 'normal_Lounge_count_mean', 'normal_Hallway_count_mean']
features:
['change_count', 'Fridge Door', 'Kitchen', 'Front Door', 'Bedroom', 'Back Door', 'Bathroom', 'Lounge', 'Hallway']


In [201]:
# Iterate through each row to create prompts

def prompt_generation(df,context_columns, feature_columns,instruction=None,
                      knowledge=None,
                      question=None,
                      answer=True,
                      INSTRUCTION_LABEL='Instruction: ',
                      KNOWLEDGE_LABEL='Knowledge: ',
                      CONTEXT_LABEL='Context: ',
                      QUESTION_LABEL='Question: ',
                      ANSWER_LABEL='Answer: '):

  prompts = []
  df['start_time'] = pd.to_datetime(df['start_time'], format="%Y-%m-%d %H:%M:%S").dt.hour.copy()  # Adjust format as needed
  df['end_time'] = pd.to_datetime(df['end_time'], format="%Y-%m-%d %H:%M:%S").dt.hour.copy()  # Adjust format as needed

  for _, row in df.iterrows():
    # Initialize prompt
      prompt = ""

      # Create context string
      context = f"The elderly {row['gender']} within the same time from {row.iloc[1]} to {row.iloc[2]} had the statistics as\n"
      context_dummy = "\n".join([f"{col.replace('_',' ').replace('normal','').replace('count','')}: {int(round(float(row[col])))}" for col in context_columns[5:]])
      context = context+context_dummy

      # Create features string
      features = "The observed counts are as\n"
      features_dummy = "\n".join([f"{col.replace('_count','')}: {int(row[col])}" for col in feature_columns])
      features = features + features_dummy

      #  Add instruction
      if instruction:
        prompt = prompt + INSTRUCTION_LABEL + instruction + '\n'

      if question:
        prompt = prompt + QUESTION_LABEL + question + '\n'

      # Construct prompt
      prompt = prompt+f"{CONTEXT_LABEL}\n{context}\nInput:\n{features}\n"

      if answer:
        answer_text = "Yes" if row['label'] == 1 else "No"
        prompt = prompt + ANSWER_LABEL + answer_text + "\n"
        #print(prompt)

      prompts.append(prompt)


  return prompts

def get_few_shot_prompt(example_prompts_list, question_prompt, question=None, instruction=None, knowledge=None,
                        INSTRUCTION_LABEL='Instruction: ',
                        KNOWLEDGE_LABEL='Knowledge: ',
                        QUESTION_LABEL='Question: ',
                        ANSWER_LABEL='Answer: '):
    """
    Constructs a few-shot learning prompt.

    Parameters:
    - example_prompts_list: List of example prompts used for in-context learning.
    - question_prompt: A prompt without an answer.
    - question: (Optional) A separate question to be included.
    - instruction: (Optional) Additional instruction.
    - knowledge: (Optional) Additional background knowledge.
    """

    few_shot_prompt = INSTRUCTION_LABEL + instruction + "\n"

    # Extract true label and remove answer from question prompt
    true_label = None
    filtered_question_prompt = ""

    for line in question_prompt.split("\n"):
        if not line.startswith(ANSWER_LABEL):
            filtered_question_prompt += line + "\n"
        else:
            true_label = line.replace(ANSWER_LABEL, '').strip()

    # Construct few-shot examples
    few_shot_prompt += "\n\n".join([f"Example {i}:\n{p}" for i, p in enumerate(example_prompts_list)])

    # Append filtered question prompt
    few_shot_prompt += f"\n\n{filtered_question_prompt.strip()}"

    # Append optional question
    if question:
        few_shot_prompt += f"\n{QUESTION_LABEL}{question}"
    else:
      # Append answer label
      few_shot_prompt += f"\n{ANSWER_LABEL}"

    return few_shot_prompt, true_label


def generate_answer(prompt, tokenizer, model,device='auto'):
    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate an answer
    output_ids = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Determine the number of prompt tokens
    prompt_token_size = len(inputs[0])
    print(prompt_token_size)

    if prompt_token_size > context_length:
      print(f'Warning! Token size ({prompt_token_size}) exceeds the context window length.')

    # Decode the output
    answer = tokenizer.decode(
        output_ids[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()
    return answer
prompts = prompt_generation(df.copy(),context_columns, feature_columns)


In [570]:
#k_shot_values = [1,5,10,20,50]
k = 5
np.random.seed(seed_id+5)
example_ids = np.random.choice(train_sample_ids,size=k,replace=False)
example_prompts = []
for i in example_ids:
  example_prompts.append(prompts[i])

In [571]:

question = "Is the elderly agitated?[Yes or No]"
instruction = "You are a psychologist who wants to detect if a person is agitated or not. You should learn from the provided examples and answer with one word [Yes/No]."

few_shot_prompt, true_label = get_few_shot_prompt(example_prompts,
                                                  prompts[test_sample_ids[41]],
                                                  question=question,
                                                  instruction=instruction)
print(few_shot_prompt)


Instruction: You are a psychologist who wants to detect if a person is agitated or not. You should learn from the provided examples and answer with one word [Yes/No].
Example 0:
Context: 
The elderly Female within the same time from 12 to 18 had the statistics as
 samples: 85
 change mean: 125
 Fridge Door  mean: 33
 Kitchen  mean: 35
 Front Door  mean: 11
 Bedroom  mean: 15
 Back Door  mean: 14
 Bathroom  mean: 4
 Lounge  mean: 40
 Hallway  mean: 38
Input:
The observed counts are as
change: 260
Fridge Door: 49
Kitchen: 38
Front Door: 4
Bedroom: 7
Back Door: 2
Bathroom: 3
Lounge: 45
Hallway: 51
Answer: Yes


Example 1:
Context: 
The elderly Male within the same time from 12 to 18 had the statistics as
 samples: 26
 change mean: 164
 Fridge Door  mean: 17
 Kitchen  mean: 39
 Front Door  mean: 14
 Bedroom  mean: 10
 Back Door  mean: 10
 Bathroom  mean: 8
 Lounge  mean: 54
 Hallway  mean: 56
Input:
The observed counts are as
change: 0
Fridge Door: 0
Kitchen: 24
Front Door: 10
Bedroom: 10


In [547]:
print(f'True label: {true_label}')


True label: No


In [203]:
# Example: Print the first prompt
#print(prompts[1])
answer = generate_answer(few_shot_prompt, tokenizer, model,device)
print(answer)
print(f'True label: {true_label}')

1014

True label: Yes


Analyze results

In [564]:
import pandas as pd

file_path = '/content/1234llm/results_in_context_learning.csv'
results_df = pd.read_csv(file_path)
results_df

,model,k,id,seed 1,seed 2,seed 3,seed 4,seed 5,true label,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,ChatGPT-4,5,0,1,1,1,1,1,1,NaN,NaN,NaN,NaN,NaN
1,ChatGPT-4,5,1,1,1,0,0,1,1,NaN,NaN,NaN,NaN,NaN
2,ChatGPT-4,5,2,1,1,1,1,1,1,NaN,NaN,NaN,NaN,NaN
3,ChatGPT-4,5,3,1,1,1,0,0,1,NaN,NaN,NaN,NaN,NaN
4,ChatGPT-4,5,4,1,1,1,1,1,1,NaN,NaN,NaN,NaN,NaN
5,ChatGPT-4,5,5,1,1,1,0,1,1,NaN,NaN,NaN,NaN,NaN
6,ChatGPT-4,5,6,1,1,1,1,1,1,NaN,NaN,NaN,NaN,NaN
7,ChatGPT-4,5,7,1,1,1,1,1,1,NaN,NaN,NaN,NaN,NaN
8,ChatGPT-4,5,8,1,1,0,1,1,1,NaN,NaN,NaN,NaN,NaN
9,ChatGPT-4,5,9,1,1,1,1,0,1,NaN,NaN,NaN,NaN,NaN


In [574]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score, precision_recall_curve, auc
import numpy as np

k = 15
result_k = results_df[results_df['k']==k]  ## !!!I should rename the column name to k

# Get the true labels
y_true = result_k['true label']

# Initialize lists to store the metrics for each seed
accuracies = []
f1_scores = []
precisions = []
sensitivities = []
specificities = []
roc_aucs = []
pr_aucs = []


# Iterate through each of the seeds
for s in [1, 2, 3, 4, 5]:
    col = f'seed {s}'
    y_pred = result_k[col]

    # Calculate Accuracy
    accuracy = accuracy_score(y_true, y_pred)
    accuracies.append(accuracy)

    # Calculate F1 Score
    f1 = f1_score(y_true, y_pred)
    f1_scores.append(f1)

    # Calculate Precision
    precision = precision_score(y_true, y_pred)
    precisions.append(precision)

    # Calculate Sensitivity (Recall)
    sensitivity = recall_score(y_true, y_pred)
    sensitivities.append(sensitivity)

    # Calculate Specificity
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)
    specificities.append(specificity)

    # Calculate ROC AUC
    roc_auc = roc_auc_score(y_true, y_pred)
    roc_aucs.append(roc_auc)

    # Calculate PR AUC
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_pred)
    pr_auc = auc(recall_curve, precision_curve)
    pr_aucs.append(pr_auc)

# Calculate average and standard deviation of the metrics
avg_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

avg_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)

avg_precision = np.mean(precisions)
std_precision = np.std(precisions)

avg_sensitivity = np.mean(sensitivities)
std_sensitivity = np.std(sensitivities)

avg_specificity = np.mean(specificities)
std_specificity = np.std(specificities)

avg_roc_auc = np.mean(roc_aucs)
std_roc_auc = np.std(roc_aucs)

avg_pr_auc = np.mean(pr_aucs)
std_pr_auc = np.std(pr_aucs)

# Print the average and std of the metrics
print(f"Average Accuracy: {avg_accuracy:.2f} ± {std_accuracy:.2f}")
print(f"Average F1 Score: {avg_f1:.2f} ± {std_f1:.2f}")
print(f"Average Precision: {avg_precision:.2f} ± {std_precision:.2f}")
print(f"Average Sensitivity (Recall): {avg_sensitivity:.2f} ± {std_sensitivity:.2f}")
print(f"Average Specificity: {avg_specificity:.2f} ± {std_specificity:.2f}")
print(f"Average PR AUC: {avg_pr_auc:.2f} ± {std_pr_auc:.2f}")
print(f"Average ROC AUC: {avg_roc_auc:.2f} ± {std_roc_auc:.2f}")


Average Accuracy: 0.55 ± 0.06
Average F1 Score: 0.65 ± 0.04
Average Precision: 0.53 ± 0.04
Average Sensitivity (Recall): 0.83 ± 0.03
Average Specificity: 0.27 ± 0.11
Average PR AUC: 0.72 ± 0.03
Average ROC AUC: 0.55 ± 0.06


Majority Voting

In [573]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score, precision_recall_curve, auc
)
import itertools
from statistics import mode, StatisticsError
from collections import Counter

k = 5
result_k = results_df[results_df['k']==k]  ## !!!I should rename the column name to k

# Get the true labels
y_true = result_k['true label']

# List of all seed columns
seeds = [f'seed 1', f'seed 2', f'seed 3', f'seed 4', f'seed 5']

# Initialize lists to store the metrics for each combination
accuracies, f1_scores, precisions, sensitivities = [], [], [], []
specificities, roc_aucs, pr_aucs = [], [], []

# Generate all combinations of 3 seeds from the 5 seed columns
seed_combinations = itertools.combinations(seeds, 3)

# Iterate through each combination of 3 seed columns
for seed_combination in seed_combinations:
    # Create an empty list to store predictions for each row
    majority_predictions = []

    # Iterate through the rows of the dataframe
    for _, row in result_k.iterrows():
        # Get the predictions from the selected seed columns
        predictions = [row[seed] for seed in seed_combination]

        # Handle majority voting, with tie-breaking
        try:
            majority_vote = mode(predictions)
        except StatisticsError:  # Handle ties
            counter = Counter(predictions)
            max_count = max(counter.values())
            modes = [key for key, count in counter.items() if count == max_count]
            majority_vote = modes[0]  # Pick the first mode in case of ties

        # Append the majority vote prediction
        majority_predictions.append(majority_vote)

    # Convert the majority_predictions list to a numpy array for easier metrics calculation
    y_pred_majority = np.array(majority_predictions)

    # Calculate Accuracy
    accuracies.append(accuracy_score(y_true, y_pred_majority))

    # Calculate F1 Score
    f1_scores.append(f1_score(y_true, y_pred_majority))

    # Calculate Precision
    precisions.append(precision_score(y_true, y_pred_majority))

    # Calculate Sensitivity (Recall)
    sensitivities.append(recall_score(y_true, y_pred_majority))

    # Calculate Specificity
    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_majority, labels=[0, 1]).ravel()
        specificities.append(tn / (tn + fp))
    except ValueError:  # Handle cases where confusion matrix fails
        specificities.append(float('nan'))

    # Calculate ROC AUC
    try:
        roc_auc = roc_auc_score(y_true, y_pred_majority)
        roc_aucs.append(roc_auc)
    except ValueError:
        roc_aucs.append(float('nan'))  # Handle edge cases where ROC AUC fails

    # Calculate PR AUC
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_pred_majority)
    pr_auc = auc(recall_curve, precision_curve)
    pr_aucs.append(pr_auc)

# Calculate average and standard deviation of the metrics over all combinations
def compute_avg_std(metrics):
    return np.mean(metrics), np.std(metrics)

avg_accuracy, std_accuracy = compute_avg_std(accuracies)
avg_f1, std_f1 = compute_avg_std(f1_scores)
avg_precision, std_precision = compute_avg_std(precisions)
avg_sensitivity, std_sensitivity = compute_avg_std(sensitivities)
avg_specificity, std_specificity = compute_avg_std(specificities)
avg_roc_auc, std_roc_auc = compute_avg_std(roc_aucs)
avg_pr_auc, std_pr_auc = compute_avg_std(pr_aucs)

# Print the average and std of the metrics
print("Performance Metrics Across Seed Combinations:")
print(f"Accuracy: {avg_accuracy:.2f} ± {std_accuracy:.2f}")
print(f"F1 Score: {avg_f1:.2f} ± {std_f1:.2f}")
print(f"Precision: {avg_precision:.2f} ± {std_precision:.2f}")
print(f"Sensitivity (Recall): {avg_sensitivity:.2f} ± {std_sensitivity:.2f}")
print(f"Specificity: {avg_specificity:.2f} ± {std_specificity:.2f}")
print(f"PR AUC: {avg_pr_auc:.2f} ± {std_pr_auc:.2f}")
print(f"ROC AUC: {avg_roc_auc:.2f} ± {std_roc_auc:.2f}")


Performance Metrics Across Seed Combinations:
Accuracy: 0.61 ± 0.02
F1 Score: 0.68 ± 0.02
Precision: 0.58 ± 0.01
Sensitivity (Recall): 0.81 ± 0.05
Specificity: 0.41 ± 0.03
PR AUC: 0.74 ± 0.02
ROC AUC: 0.61 ± 0.02
